# Barttorvik Full Season

Get the Barttorvik ratings from previous full seasons (regular season plus postseason) for features about previous team strength. Assumes all data has been copied from website (https://barttorvik.com/#) to Excel, as specified in the steps below. 

Steps before this file:

1. Copy from the website, excluding the initial row with D1 averages
2. Make sure the REC column in excel is text only before pasting (otherwise it tries to convert the records into a date)
3. Paste into excel with "Match Destination Formatting"
4. Save as csv into the folder

Note: Do not copy from previous year's project because the team naming conventions sometimes switch and this file relies on consistent names across seasons (i.e. North Carolina State in 2024 to N.C. State in 2025)

In [1]:
SEASON = 2025

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 100)

def load_bartorvik(season: int):
    d = pd.read_csv(f"../data/unprocessed/mens_barttorvik_full_season/barttorvik_full_season_{season}.csv")
    d.columns = d.columns.str.upper()
    d['Season'] = season

    return d

df = pd.concat(
    [
        load_bartorvik(season)
        for season in range(2008, SEASON)
    ],
    ignore_index=True
)

df['RK'] = pd.to_numeric(df['RK'], errors='coerce')

df = df.loc[df['RK'].notna(), ['Season', 'TEAM', 'BARTHAG']].reset_index(drop=True)

df

,Season,TEAM,BARTHAG
0,2008,Kansas,0.9815
1,2008,Memphis,0.9703
2,2008,UCLA,0.9647
3,2008,North Carolina,0.9643
4,2008,Wisconsin,0.958
...,...,...,...
5955,2024,VMI,0.0654
5956,2024,IU Indy,0.0635
5957,2024,Saint Francis,0.0634
5958,2024,Coppin St.,0.045


Some teams (like Ivy League in 2021) are missing, so add them in with NAs

In [3]:
df_full_teams = pd.DataFrame(
    [(season, team) for team in df['TEAM'].unique() for season in df['Season'].unique()],
    columns=['Season', 'TEAM']
)

df_full_teams

,Season,TEAM
0,2008,Kansas
1,2009,Kansas
2,2010,Kansas
3,2011,Kansas
4,2012,Kansas
...,...,...
6234,2020,Le Moyne
6235,2021,Le Moyne
6236,2022,Le Moyne
6237,2023,Le Moyne


In [4]:
df = (
    pd.merge(
        df, 
        df_full_teams,
        how='right',
        on=['Season', 'TEAM']
    )
    .sort_values(
        ['Season', 'TEAM'], 
        ignore_index=True
    )
)

df

,Season,TEAM,BARTHAG
0,2008,Abilene Christian,NaN
1,2008,Air Force,0.5629
2,2008,Akron,0.7515
3,2008,Alabama,0.7555
4,2008,Alabama A&M,0.1273
...,...,...,...
6234,2024,Wright St.,0.553
6235,2024,Wyoming,0.53
6236,2024,Xavier,0.7997
6237,2024,Yale,0.7227


In [5]:
df['Past 4 Years BARTHAG'] = (
    df
    .groupby(['TEAM'])
    ['BARTHAG']
    .rolling(window=4, min_periods=2)  # at least 2 years of data to calculate
    .mean()
    .reset_index()
    .set_index('level_1')
)['BARTHAG']

df.rename(
    columns={
    'BARTHAG': 'Past Year BARTHAG'
    }, 
    inplace=True
)

df['Season'] += 1  # shift by a year so BARTHAGs are from past instead of the current rating

df = df.loc[df['Season'] >= 2012, :].reset_index(drop=True)

df

,Season,TEAM,Past Year BARTHAG,Past 4 Years BARTHAG
0,2012,Abilene Christian,NaN,NaN
1,2012,Air Force,0.5782,0.462825
2,2012,Akron,0.6049,0.677650
3,2012,Alabama,0.8419,0.780925
4,2012,Alabama A&M,0.1283,0.103600
...,...,...,...,...
5133,2025,Wright St.,0.553,0.551775
5134,2025,Wyoming,0.53,0.590825
5135,2025,Xavier,0.7997,0.825800
5136,2025,Yale,0.7227,0.673600


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5138 entries, 0 to 5137
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Season                5138 non-null   int64  
 1   TEAM                  5138 non-null   object 
 2   Past Year BARTHAG     4928 non-null   object 
 3   Past 4 Years BARTHAG  4928 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 160.7+ KB


In [ ]:
df.to_csv(f'../data/preprocessed/mens_barttorvik_full_season/barttorvik_full_season.csv', index=False)

'Done'

'Done'